# LLM10 Unbounded Consumption — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM10 — Unbounded Consumption | **Risk Severity**: Medium

This notebook:
1. **Uploads** all artifact files (scenarios, checks, drivers) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM10 unbounded consumption test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks/drivers upsert).
Registered IDs are available in-memory for the evaluation steps below.

In [1]:
%pip install okareo python-dotenv --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_check_md,
    parse_driver_md,
    parse_check_py_meta,
    parse_check_py_metadata,
    parse_check_py,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


✓ Okareo SDK initialized (key: ...a7sKA)
Category directory: /Users/guiair/dev/okareo/compliance-owasp/owasp/LLM10-unbounded-consumption


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [3]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}  # name -> scenario object

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM10-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

Uploading scenario: LLM10-infinite-loop from infinite-loop.jsonl
  ✓ Registered: LLM10-infinite-loop (ID: 4e66fb63-8532-402c-8620-4704279d01bc)
Uploading scenario: LLM10-resource-exhaustion from resource-exhaustion.jsonl
  ✓ Registered: LLM10-resource-exhaustion (ID: 839f2758-8c28-4c13-96f1-9b41aa68f370)

Total scenarios uploaded: 2


### Register Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [4]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}  # name -> check id

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_check_md(md_path)
    print(f"Registering check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


Registering check: LLM10-loop-detection-check from loop-detection-check.md
  ✓ Registered: LLM10-loop-detection-check (ID: b0c126e7-a22f-420e-951c-7d1ea508e13d)
Registering check: LLM10-resource-policy-enforcement-check from resource-policy-enforcement-check.md
  ✓ Registered: LLM10-resource-policy-enforcement-check (ID: 2cf7bb30-af0c-4b78-aef5-d6e510ebf937)

Total checks registered: 2


### Register Drivers

Scans `drivers/` for `.md` files, parses YAML front matter and persona prompt,
and registers each via `create_or_update_driver` using a `Driver` object.

In [5]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}  # name -> driver data dict

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_driver_md(md_path)
    print(f"Registering driver: {driver_data['name']} from {md_path.name}")

    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ Registered: {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")


Registering driver: LLM10-loop-inducing-driver from loop-inducing-driver.md
  ✓ Registered: LLM10-loop-inducing-driver (ID: a4987cf3-acae-4b3d-852e-d9e05d151b1e)
Registering driver: LLM10-resource-exhaustion-driver from resource-exhaustion-driver.md
  ✓ Registered: LLM10-resource-exhaustion-driver (ID: 8a73d9e6-7499-45c4-a60d-06dd7aec4797)

Total drivers registered: 2


### Artifact Upload Summary

In [6]:
print("=" * 60)
print("LLM10 Unbounded Consumption — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print(f"\nDrivers ({len(registered_drivers)}):")
for name, drv in registered_drivers.items():
    print(f"  • {name} → {drv.id}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

LLM10 Unbounded Consumption — Artifact Upload Summary

Scenarios (2):
  • LLM10-infinite-loop → 4e66fb63-8532-402c-8620-4704279d01bc
  • LLM10-resource-exhaustion → 839f2758-8c28-4c13-96f1-9b41aa68f370

Checks (2):
  • LLM10-loop-detection-check → b0c126e7-a22f-420e-951c-7d1ea508e13d
  • LLM10-resource-policy-enforcement-check → 2cf7bb30-af0c-4b78-aef5-d6e510ebf937

Drivers (2):
  • LLM10-loop-inducing-driver → a4987cf3-acae-4b3d-852e-d9e05d151b1e
  • LLM10-resource-exhaustion-driver → 8a73d9e6-7499-45c4-a60d-06dd7aec4797

✓ All artifacts ready. Proceeding to evaluation...


---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.env` file (copy `owasp/target.env.example` and fill in your values).
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

The target is registered as a `CustomEndpointTarget` with `TurnConfig` defining how to send messages.
LLM10 scenarios use `okareo.run_simulation()` with `max_turns=10`. Scenario 1 (infinite-loop) uses `first_turn="target"`;
Scenario 2 (resource-exhaustion) uses `first_turn="driver"`. Each scenario is paired with its own driver and check.

In [7]:
# Target loaded from owasp/target.env. To use a different config: target = build_target(CATEGORY_DIR, env_path="target.prod.env")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

MAX_TURNS = 10

SCENARIO_DRIVER_MAP = {
    "LLM10-infinite-loop":         "LLM10-loop-inducing-driver",
    "LLM10-resource-exhaustion":   "LLM10-resource-exhaustion-driver",
}

SCENARIO_FIRST_TURN = {
    "LLM10-infinite-loop":         "target",
    "LLM10-resource-exhaustion":   "driver",
}

SCENARIO_CHECK_MAP = {
    "LLM10-infinite-loop":         ["LLM10-loop-detection-check"],
    "LLM10-resource-exhaustion":   ["LLM10-resource-policy-enforcement-check"],
}

✓ Target agent: FinanceBot


### Build Target

Constructs a `CustomEndpointTarget` from `owasp/target.env` using `TurnConfig` for the
next-turn endpoint and optional `SessionConfig` / `EndSessionConfig` for session management.

In [8]:
# Target built in config cell above via build_target(CATEGORY_DIR)


### Multi-Turn Simulations — Both LLM10 Scenarios

Each scenario runs via `okareo.run_simulation()` with `max_turns=10`.
Scenario 1 (infinite-loop) uses `first_turn="target"`; Scenario 2 (resource-exhaustion) uses `first_turn="driver"`.
Each scenario is paired with its dedicated driver and check via `SCENARIO_DRIVER_MAP` and `SCENARIO_CHECK_MAP`.

In [9]:
simulation_results = {}  # scenario_name -> test run result

for scenario_name, driver_name in SCENARIO_DRIVER_MAP.items():
    if scenario_name not in registered_scenarios:
        print(f"⚠ Scenario {scenario_name} not found in registered scenarios — skipping")
        continue

    if driver_name not in registered_drivers:
        print(f"⚠ Driver {driver_name} not found in registered drivers — skipping {scenario_name}")
        continue

    print(f"\n{'='*60}")
    print(f"Running simulation: {scenario_name}")
    print(f"Driver: {driver_name} | Max turns: {MAX_TURNS}")
    print(f"{'='*60}")

    try:
        driver_reg = registered_drivers[driver_name]
        multi_turn_driver = Driver(
            temperature=driver_reg.temperature if hasattr(driver_reg, "temperature") else 0.7,
            name=driver_name,
            prompt_template=driver_reg.prompt_template,
        )

        scenario = registered_scenarios[scenario_name]

        simulation_run = okareo.run_simulation(
            target=target,
            driver=multi_turn_driver,
            name=f"LLM10 Simulation — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn=SCENARIO_FIRST_TURN.get(scenario_name, "target"),
            scenario=scenario,
            max_turns=MAX_TURNS,
            checks=SCENARIO_CHECK_MAP.get(scenario_name, []),
        )
        simulation_results[scenario_name] = simulation_run
        print(f"  ✓ Simulation complete: {simulation_run.id}")
        if hasattr(simulation_run, "app_link") and simulation_run.app_link:
            print(f"  View: {simulation_run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        simulation_results[scenario_name] = None


Running simulation: LLM10-infinite-loop
Driver: LLM10-loop-inducing-driver | Max turns: 10
  ✓ Simulation complete: be5762e0-5b10-4053-b2c5-fb53cddfeccf
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/be5762e0-5b10-4053-b2c5-fb53cddfeccf

Running simulation: LLM10-resource-exhaustion
Driver: LLM10-resource-exhaustion-driver | Max turns: 10
  ✓ Simulation complete: 9c53650f-0e6e-4a4a-8120-f772f5e51b28
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/9c53650f-0e6e-4a4a-8120-f772f5e51b28


### Results Summary

In [10]:
print("\n" + "=" * 60)
print("LLM10 UNBOUNDED CONSUMPTION — EVALUATION RESULTS")
print("OWASP Category: LLM10 | Risk Severity: Medium")
print("=" * 60)

print(f"\n{'Scenario':<46} {'Status':<10} {'Link / Run ID'}")
print("-" * 110)
for name, result in simulation_results.items():
    if result is None:
        print(f"{name:<46} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<46} {'COMPLETE':<10} {link}")

errors = sum(1 for r in simulation_results.values() if r is None)
print(f"\nTotal evaluated: {len(simulation_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")


LLM10 UNBOUNDED CONSUMPTION — EVALUATION RESULTS
OWASP Category: LLM10 | Risk Severity: Medium

Scenario                                       Status     Link / Run ID
--------------------------------------------------------------------------------------------------------------
LLM10-infinite-loop                            COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/be5762e0-5b10-4053-b2c5-fb53cddfeccf
LLM10-resource-exhaustion                      COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/9c53650f-0e6e-4a4a-8120-f772f5e51b28

Total evaluated: 2 | Errors: 0
✓ All scenarios completed. See Okareo dashboard for full results.


### Detailed Results (Optional)

Retrieve per-row scores and conversation transcripts for any completed simulation run.

In [11]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)